<div class="doris-cover">
  <div class="doris-cover-kicker">DEMO 02 · DBT × APACHE DORIS</div>
  <div class="doris-cover-title">客户地域分析</div>
  <p class="doris-cover-lead">关联两个 Doris Database 中的客户地址和订单，生成每个州的客户、订单和收入指标。</p>
  <span class="doris-cover-note">Cross-database Source · View · ref() · Table · Data Test</span>
</div>

## 1. 检查执行环境

先运行下面的单元格。它会使用 Demo 环境中的 dbt 和当前 Doris 连接参数，并确认 Backend 可用。

In [ ]:
import importlib.util
from pathlib import Path


def find_demo_dir(start):
    for candidate in (start, *start.parents):
        demo_dir = candidate / "examples/doris-demos"
        if demo_dir.is_dir():
            return demo_dir
    raise FileNotFoundError("请从 dbt-for-apache-doris 仓库目录或其子目录启动 Jupyter。")


demo_dir = find_demo_dir(Path.cwd().resolve())
helper_path = demo_dir / "scripts/notebook_helpers.py"
helper_spec = importlib.util.spec_from_file_location("dbt_doris_notebook_helpers", helper_path)
notebook_helpers = importlib.util.module_from_spec(helper_spec)
helper_spec.loader.exec_module(notebook_helpers)

runner = notebook_helpers.DemoRunner()
runner.show_environment()

## 2. Demo 2：客户地域分析

区域运营需要判断客户和收入集中在哪些州，以安排市场活动、客户运营和销售覆盖。这个 Demo 使用客户默认收货地址与有效订单，生成一张每州一行的地域经营表。

<table class="doris-index">
  <tr><th>业务使用者</th><td>区域运营、市场团队、销售管理</td></tr>
  <tr><th>核心问题</th><td>各州有多少客户和有效订单，贡献多少收入，客户价值和下单频次如何？</td></tr>
  <tr><th>统计口径</th><td>客户按默认收货地址归属州；订单只保留 COMPLETED、DELIVERED、SHIPPED</td></tr>
  <tr><th>交付结果</th><td>州级指标表 <code>fct_state_customers</code></td></tr>
</table>

下面展示两张 Doris Source 如何经过 staging View，再通过 `ref()` 关联成州级指标。

<div class="doris-flow">
  <div class="doris-flow-step"><strong>两个 Source</strong>地址表 + 订单表</div><div class="doris-flow-arrow">→</div>
  <div class="doris-flow-step"><strong>Staging Views</strong>过滤默认地址和有效订单</div><div class="doris-flow-arrow">→</div>
  <div class="doris-flow-step"><strong><code>ref()</code> Join</strong>按客户关联两条链路</div><div class="doris-flow-arrow">→</div>
  <div class="doris-flow-step"><strong>州级指标</strong><code>fct_state_customers</code></div>
</div>

### 2.1 准备并查看两张源表

源数据放在两个 Doris Database：地址表有 4 条记录，其中 1 条为非默认收货地址；订单表有 4 条记录，其中 1 条为取消订单。

In [ ]:
geo_dir = runner.examples_root / "doris-demos/geographic"
runner.show_file("Fixture SQL", geo_dir / "scripts/setup.sql")
runner.run_sql_file("创建地域分析源表", geo_dir / "scripts/setup.sql")
runner.query("输入：客户地址", """
select address_id, customer_id, state_province, is_default_shipping
from dbt_demo_geographic_customer.CUSTOMER_ADDRESSES
order by address_id
""")
runner.query("输入：订单", """
select order_id, customer_id, grand_total, status
from dbt_demo_geographic_orders.ORDERS
order by order_id
""")

### 2.2 创建 staging View

两个 staging Model 分别使用 `source()` 读取源表：地址 Model 只保留默认收货地址，订单 Model 只保留 COMPLETED、DELIVERED、SHIPPED。

In [ ]:
runner.show_file("地址 staging Model", geo_dir / "models/stg_customer_addresses.sql")
runner.show_file("订单 staging Model", geo_dir / "models/stg_orders.sql")
runner.show_file("Source 声明", geo_dir / "models/sources.yml")
runner.run_dbt("创建两个 staging View", geo_dir, "run", "--select", "stg_customer_addresses", "stg_orders")
runner.query("中间结果：staging View", """
select 'address' as stage, cast(address_id as string) as record_id, state_province as value
from dbt_demo_geographic.stg_customer_addresses
union all
select 'order', cast(order_id as string), cast(grand_total as string)
from dbt_demo_geographic.stg_orders
order by stage, record_id
""")

### 2.3 通过 `ref()` 生成州级指标

事实 Model 通过两个 staging View join。结果同时给出州级规模、客单价、客均收入和客均订单数：CA 有 2 个客户、2 个有效订单、145.00 收入，客单价和客均收入均为 72.50；NY 有 1 个客户、1 个有效订单、50.00 收入。

In [ ]:
runner.show_file("州级指标 Model", geo_dir / "models/fct_state_customers.sql")
runner.run_dbt("创建 fct_state_customers", geo_dir, "run", "--select", "fct_state_customers")
runner.query("输出：州级客户、订单、收入与人均指标", """
select state_province, customer_count, order_count, total_revenue,
       avg_order_value, revenue_per_customer, orders_per_customer
from dbt_demo_geographic.fct_state_customers
order by state_province
""")

### 2.4 执行 Data Test 并验证对象类型

Data Test 检查州名和客户数非空；verifier 还检查最终对象是 Table，两个中间对象是 View。

In [ ]:
runner.show_file("Data Test 定义", geo_dir / "models/geographic.yml")
runner.run_dbt("验证 fct_state_customers", geo_dir, "test", "--select", "fct_state_customers")
runner.run_script("校验地域 Demo", geo_dir / "scripts/verify.sh")
runner.query("最终对象类型", """
select table_name, table_type
from information_schema.tables
where table_schema = 'dbt_demo_geographic'
order by table_name
""")

## 完成

两张 Source、两个 staging View、州级指标 Table 和 Data Test 均已通过校验。